# Validacion walk-forward (5 origenes temporales, distintos regimenes hidrologicos)

Tarea de ruta critica prometida en el Anexo 1 (Fase 2, OE2): "verificando la estabilidad de los
hiperparametros elegidos en 5 origenes temporales que abarcan distintos regimenes hidrologicos".

En vez de un unico corte train/test (2019-2025 / 2026), se repite el entrenamiento y evaluacion en
5 origenes distintos, cada uno cayendo en un regimen ENSO diferente segun data/external/oni_index.csv.
Cada origen es un escenario "ventana creciente": se entrena con todo lo disponible hasta el corte, y
se evalua en los 12 meses inmediatamente posteriores -- asi se simula la situacion real de re-entrenar
periodicamente con el historial que se tiene en ese momento. Hasta el 2026-09-11 la prueba de cada origen era
de 3 meses; se amplio a un anio completo, el minimo que recomienda Lago et al. (2021).

Se reutilizan las mismas features ya construidas en 05_features_compartidas_juan.ipynb
(dataset_features_2019_2025.csv + dataset_features_2026.csv), y la misma configuracion final
de XGBoost (06_modelo_xgboost_juan.ipynb) y Prophet (04_modelo_prophet_juan.ipynb) -- aqui no se
vuelve a tunear nada, solo se mide que tan estable es lo ya elegido.

In [1]:
# --- Celda de arranque ---
import pandas as pd
import numpy as np
from pathlib import Path

def encontrar_raiz_proyecto(marcador="requirements.txt"):
    actual = Path.cwd()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / marcador).exists():
            return carpeta
    raise FileNotFoundError(f"No encontre '{marcador}' subiendo desde {actual}")

RAIZ = encontrar_raiz_proyecto()
print("Raiz del proyecto:", RAIZ)

def calcular_metricas(y_real, y_pred):
    y_real, y_pred = np.asarray(y_real, dtype=float), np.asarray(y_pred, dtype=float)
    error = y_real - y_pred
    mae = np.abs(error).mean()
    rmse = np.sqrt((error ** 2).mean())
    mape = (np.abs(error) / y_real).mean() * 100
    return mae, rmse, mape


Raiz del proyecto: C:\Users\mgdbj\xm-spot-price-predictor


In [2]:
# --- Cargar el dataset de features ya construido (mismo insumo que 04 y 06) ---
df_train = pd.read_csv(RAIZ / "data" / "processed" / "dataset_features_2019_2025.csv", parse_dates=["fecha_hora"])
df_test = pd.read_csv(RAIZ / "data" / "processed" / "dataset_features_2026.csv", parse_dates=["fecha_hora"])

df_completo = pd.concat([df_train, df_test], ignore_index=True).sort_values("fecha_hora").reset_index(drop=True)
print("Rango disponible:", df_completo["fecha_hora"].min(), "a", df_completo["fecha_hora"].max())
print("Filas:", df_completo.shape[0])


Rango disponible: 2019-01-31 23:00:00 a 2026-08-05 23:00:00
Filas: 65833


In [3]:
# --- Origenes del walk-forward: 5 ventanas CONSECUTIVAS de 12 meses (un anio completo de prueba cada una) + 2026 ---
# Cambio 2026-09-11 (pedido del usuario): antes eran ventanas de 3 meses, por debajo del minimo de 1 anio de prueba
# que recomienda Lago et al. (2021). Se usan anios julio->junio en vez de anios calendario para que el Origen 1
# entrene con la pandemia ya incluida (con anios calendario entrenaria solo con 2019) y para que el episodio
# completo de El Nino 2023-24 quede dentro de un solo origen. Las 5 ventanas no se solapan.
# El Origen 6 (2026, el anio de publicacion) no cambia.
# ONI promedio [rango] de cada ventana:
#   Origen 1  2020-07..2021-06:  -0.71 [-1.1, -0.2]  La Nina
#   Origen 2  2021-07..2022-06:  -0.70 [-0.9, -0.3]  La Nina (continuacion)
#   Origen 3  2022-07..2023-06:  -0.42 [-0.9, +0.7]  La Nina triple-dip -> neutral
#   Origen 4  2023-07..2024-06:  +1.30 [+0.2, +2.0]  El Nino fuerte (episodio completo)
#   Origen 5  2024-07..2025-06:  -0.13 [-0.5, +0.2]  neutral / La Nina debil
origenes = [
    {"nombre": "Origen 1", "regimen": "La Nina 2020-21",                     "corte_train": "2020-07-01", "test_inicio": "2020-07-01", "test_fin": "2021-06-30 23:00"},
    {"nombre": "Origen 2", "regimen": "La Nina 2021-22",                     "corte_train": "2021-07-01", "test_inicio": "2021-07-01", "test_fin": "2022-06-30 23:00"},
    {"nombre": "Origen 3", "regimen": "La Nina triple-dip -> neutral",       "corte_train": "2022-07-01", "test_inicio": "2022-07-01", "test_fin": "2023-06-30 23:00"},
    {"nombre": "Origen 4", "regimen": "El Nino 2023-24 (episodio completo)", "corte_train": "2023-07-01", "test_inicio": "2023-07-01", "test_fin": "2024-06-30 23:00"},
    {"nombre": "Origen 5", "regimen": "Neutral / La Nina debil 2024-25",     "corte_train": "2024-07-01", "test_inicio": "2024-07-01", "test_fin": "2025-06-30 23:00"},
    {"nombre": "Origen 6", "regimen": "El Nino 2026 (neutral->fuerte)",      "corte_train": "2026-01-01", "test_inicio": "2026-01-01", "test_fin": "2026-08-05"},
]

for o in origenes:
    ventana = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])]
    o["oni_min"] = ventana["oni"].min()
    o["oni_max"] = ventana["oni"].max()
    o["n_train"] = (df_completo["fecha_hora"] < o["corte_train"]).sum()
    o["n_test"] = len(ventana)
    print(f"{o['nombre']:10s} [{o['regimen']:24s}] train<{o['corte_train']}  test {o['test_inicio']}..{o['test_fin']}  "
          f"ONI [{o['oni_min']:.1f}, {o['oni_max']:.1f}]  n_train={o['n_train']}  n_test={o['n_test']}")


Origen 1   [La Nina 2020-21         ] train<2020-07-01  test 2020-07-01..2021-06-30 23:00  ONI [-1.1, -0.2]  n_train=12385  n_test=8760
Origen 2   [La Nina 2021-22         ] train<2021-07-01  test 2021-07-01..2022-06-30 23:00  ONI [-0.9, -0.3]  n_train=21145  n_test=8760
Origen 3   [La Nina triple-dip -> neutral] train<2022-07-01  test 2022-07-01..2023-06-30 23:00  ONI [-0.9, 0.7]  n_train=29905  n_test=8760
Origen 4   [El Nino 2023-24 (episodio completo)] train<2023-07-01  test 2023-07-01..2024-06-30 23:00  ONI [0.2, 2.0]  n_train=38665  n_test=8784
Origen 5   [Neutral / La Nina debil 2024-25] train<2024-07-01  test 2024-07-01..2025-06-30 23:00  ONI [-0.5, 0.2]  n_train=47449  n_test=8760
Origen 6   [El Nino 2026 (neutral->fuerte)] train<2026-01-01  test 2026-01-01..2026-08-05  ONI [-0.6, 1.4]  n_train=60625  n_test=5185


In [4]:
# --- Mismas features que 06_modelo_xgboost_juan.ipynb (exclusion identica) ---
columnas_excluir = ["fecha_hora", "precio_bolsa", "demanda", "generacion",
                     "anio", "mes", "hora", "dia_semana", "dia_anio"]
columnas_features = [c for c in df_completo.columns if c not in columnas_excluir]

# Mismos regresores que 04_modelo_prophet_juan.ipynb (version final corregida)
columnas_regresoras_prophet = ["aportes_hidricos", "volumen_embalses", "oni", "es_pandemia", "precio_lag24h"]

print(f"{len(columnas_features)} features para XGBoost.")
print(f"{len(columnas_regresoras_prophet)} regresores para Prophet.")


40 features para XGBoost.
5 regresores para Prophet.


In [5]:
# --- Walk-forward: XGBoost (misma config final: depth=3, lr=0.01) + persistencia ---
import xgboost as xgb

resultados = []

for o in origenes:
    train = df_completo[df_completo["fecha_hora"] < o["corte_train"]].dropna(subset=columnas_features)
    test = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])]

    X_train, y_train = train[columnas_features], np.log(train["precio_bolsa"])
    X_test, y_test = test[columnas_features], test["precio_bolsa"]

    modelo = xgb.XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.01,
                               subsample=0.8, colsample_bytree=0.8, random_state=42)
    modelo.fit(X_train, y_train)
    y_pred = np.exp(modelo.predict(X_test))

    mae, rmse, mape = calcular_metricas(y_test, y_pred)
    resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": "XGBoost",
                        "mae": mae, "rmse": rmse, "mape": mape})

    # Persistencia: precio_lag24h ya es exactamente eso (precio_bolsa desplazado 24h)
    mae_p, rmse_p, mape_p = calcular_metricas(test["precio_bolsa"], test["precio_lag24h"])
    resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": "Persistencia",
                        "mae": mae_p, "rmse": rmse_p, "mape": mape_p})

    print(f"{o['nombre']} [{o['regimen']}]  XGBoost MAE {mae:6.2f} RMSE {rmse:6.2f} MAPE {mape:5.2f}%   |  "
          f"Persistencia MAE {mae_p:6.2f} RMSE {rmse_p:6.2f} MAPE {mape_p:5.2f}%")


Origen 1 [La Nina 2020-21]  XGBoost MAE  18.25 RMSE  27.32 MAPE 11.52%   |  Persistencia MAE  16.80 RMSE  29.50 MAPE 10.16%


Origen 2 [La Nina 2021-22]  XGBoost MAE  24.14 RMSE  47.51 MAPE 11.57%   |  Persistencia MAE  17.52 RMSE  39.53 MAPE  9.20%


Origen 3 [La Nina triple-dip -> neutral]  XGBoost MAE  59.60 RMSE  99.91 MAPE 16.59%   |  Persistencia MAE  39.75 RMSE  72.89 MAPE 14.25%


Origen 4 [El Nino 2023-24 (episodio completo)]  XGBoost MAE 133.99 RMSE 196.83 MAPE 22.25%   |  Persistencia MAE  67.19 RMSE 113.01 MAPE 16.35%


Origen 5 [Neutral / La Nina debil 2024-25]  XGBoost MAE 135.47 RMSE 318.79 MAPE 20.36%   |  Persistencia MAE 101.25 RMSE 240.22 MAPE 20.67%


Origen 6 [El Nino 2026 (neutral->fuerte)]  XGBoost MAE  61.39 RMSE 107.64 MAPE 15.97%   |  Persistencia MAE  56.40 RMSE 112.33 MAPE 15.81%


**Prophet excluido de esta corrida.** En todas las pruebas anteriores (holdout 2026, walk-forward
original, ablaciones de interpolacion) fue consistentemente el modelo mas debil -- nunca le gano a
la persistencia, ni una sola vez, en ninguna prueba. Cada fit de Prophet toma ~20-90s dependiendo
del tamano de train; con 6 origenes el costo no se justifica dado el patron ya establecido.

In [6]:
# --- Walk-forward: ARX+GARCH (misma especificacion final que 08_modelo_arima_garch_juan.ipynb) ---
from arch import arch_model

regresoras_arx = ["aportes_hidricos", "volumen_embalses", "oni", "es_pandemia",
                   "hora_sin", "hora_cos", "dia_semana_sin", "dia_semana_cos"]
columnas_x_arx = regresoras_arx + ["precio_lag24h_log"]

for o in origenes:
    train = df_completo[df_completo["fecha_hora"] < o["corte_train"]].dropna(subset=columnas_features).copy()
    test = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])].copy()

    train["precio_lag24h_log"] = np.log(train["precio_lag24h"])
    test["precio_lag24h_log"] = np.log(test["precio_lag24h"])

    # Estandarizacion con estadisticos de ESTE origen unicamente (nada de 2026 ni de origenes futuros)
    y_train_log = np.log(train["precio_bolsa"])
    y_mean, y_std = y_train_log.mean(), y_train_log.std()
    y_train = (y_train_log - y_mean) / y_std * 10

    X_train_raw = train[columnas_x_arx]
    x_mean, x_std = X_train_raw.mean(), X_train_raw.std()
    X_train = (X_train_raw - x_mean) / x_std

    modelo = arch_model(y_train, x=X_train, mean="ARX", lags=0, vol="GARCH", p=1, q=1, dist="normal")
    resultado = modelo.fit(disp="off", options={"maxiter": 500})

    X_test = (test[columnas_x_arx] - x_mean) / x_std
    params_media = resultado.params[["Const"] + columnas_x_arx]
    y_pred_escalado = params_media["Const"] + (X_test * params_media[columnas_x_arx]).sum(axis=1)
    y_pred = np.exp(y_pred_escalado / 10 * y_std + y_mean)

    mae, rmse, mape = calcular_metricas(test["precio_bolsa"].values, y_pred.values)
    resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": "ARX+GARCH",
                        "mae": mae, "rmse": rmse, "mape": mape})

    print(f"{o['nombre']} [{o['regimen']}]  ARX+GARCH MAE {mae:6.2f} RMSE {rmse:6.2f} MAPE {mape:5.2f}%  "
          f"(convergencia={resultado.convergence_flag})")


Origen 1 [La Nina 2020-21]  ARX+GARCH MAE  48.60 RMSE  60.87 MAPE 29.74%  (convergencia=0)


Origen 2 [La Nina 2021-22]  ARX+GARCH MAE  19.51 RMSE  39.08 MAPE  9.88%  (convergencia=0)


Origen 3 [La Nina triple-dip -> neutral]  ARX+GARCH MAE  47.59 RMSE  74.89 MAPE 15.23%  (convergencia=0)


Origen 4 [El Nino 2023-24 (episodio completo)]  ARX+GARCH MAE  78.12 RMSE 113.53 MAPE 16.92%  (convergencia=0)


C:\Users\mgdbj\AppData\Local\Temp\ipykernel_9032\3632789593.py:25: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  resultado = modelo.fit(disp="off", options={"maxiter": 500})


Origen 5 [Neutral / La Nina debil 2024-25]  ARX+GARCH MAE 102.59 RMSE 234.56 MAPE 20.47%  (convergencia=8)


Origen 6 [El Nino 2026 (neutral->fuerte)]  ARX+GARCH MAE  55.83 RMSE 110.08 MAPE 15.48%  (convergencia=0)


### Walk-forward: N-BEATSx y N-HiTS

Mismos 6 origenes, misma logica de reentrenar desde cero por origen (consistente con como se trato
a XGBoost y ARX+GARCH -- a diferencia de los modelos neuronales que normalmente se entrenan una vez
y se evaluan con cross_validation, aqui se prioriza la comparabilidad metodologica con el resto del
walk-forward por encima de la practica estandar de neuralforecast). Mismas exogenas y misma
configuracion que `09_modelos_deep_learning_juan.ipynb` (input_size=168h, confirmado por su propio
grid search), incluyendo la estandarizacion de exogenas -- pero calculada con SOLO los datos de
train de cada origen, para no filtrar estadisticos de un origen a otro.

In [7]:
# --- Walk-forward: N-BEATSx y N-HiTS ---
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATSx, NHITS
import time

hist_exog_dl = ["volumen_embalses", "aportes_hidricos", "demanda_lag24h"]
futr_exog_continua_dl = ["oni"]
futr_exog_ya_escalada_dl = ["hora_sin", "hora_cos", "dia_semana_sin", "dia_semana_cos"]
futr_exog_dl = futr_exog_continua_dl + futr_exog_ya_escalada_dl
INPUT_SIZE_DL = 168
MAX_STEPS_DL = 1000

for o in origenes:
    train = df_completo[df_completo["fecha_hora"] < o["corte_train"]].dropna(subset=columnas_features).copy()
    test = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])].copy()

    df_origen = pd.concat([train, test], ignore_index=True).sort_values("fecha_hora").reset_index(drop=True)
    mascara_train_o = df_origen["fecha_hora"] < o["corte_train"]

    df_nf_o = df_origen[["fecha_hora", "precio_bolsa"] + hist_exog_dl + futr_exog_dl].copy()
    for col in hist_exog_dl + futr_exog_continua_dl:
        mu, sigma = df_nf_o.loc[mascara_train_o, col].mean(), df_nf_o.loc[mascara_train_o, col].std()
        df_nf_o[col] = (df_nf_o[col] - mu) / sigma

    df_nf_o["unique_id"] = "precio_bolsa"
    df_nf_o = df_nf_o.rename(columns={"fecha_hora": "ds", "precio_bolsa": "y"})
    df_nf_o = df_nf_o[["unique_id", "ds", "y"] + hist_exog_dl + futr_exog_dl]

    n_test_o = int((~mascara_train_o).sum())
    n_windows_o = n_test_o // 24

    m_nbeatsx = NBEATSx(h=24, input_size=INPUT_SIZE_DL, hist_exog_list=hist_exog_dl, futr_exog_list=futr_exog_dl,
                          max_steps=MAX_STEPS_DL, val_check_steps=100, random_seed=42, enable_progress_bar=False)
    m_nhits = NHITS(h=24, input_size=INPUT_SIZE_DL, hist_exog_list=hist_exog_dl, futr_exog_list=futr_exog_dl,
                      max_steps=MAX_STEPS_DL, val_check_steps=100, random_seed=42, enable_progress_bar=False)
    nf_o = NeuralForecast(models=[m_nbeatsx, m_nhits], freq="h")

    t0 = time.time()
    cv_o = nf_o.cross_validation(df=df_nf_o, n_windows=n_windows_o, step_size=24)
    dt = time.time() - t0

    for col_modelo, nombre_modelo in [("NBEATSx", "N-BEATSx"), ("NHITS", "N-HiTS")]:
        mae, rmse, mape = calcular_metricas(cv_o["y"].values, cv_o[col_modelo].values)
        resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": nombre_modelo,
                            "mae": mae, "rmse": rmse, "mape": mape})

    print(f"{o['nombre']} [{o['regimen']}]  N-BEATSx/N-HiTS listos ({dt:.0f}s, n_windows={n_windows_o})")


C:\Users\mgdbj\xm-spot-price-predictor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-11 22:50:16,867	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-09-11 22:50:17,209	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Seed set to 42


Seed set to 42


...


  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 4.3 M  | train
--------------------------------------------------------------
4.3 M     Trainable params
0         Non-trainable params
4.3 M     Total params
17.267    Total estimated model params size (MB)
37        Modules in train mode
0         Modules in eval mode


`Trainer.fit` stopped: `max_steps=1000` reached.


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Origen 6 [El Nino 2026 (neutral->fuerte)]  N-BEATSx/N-HiTS listos (255s, n_windows=216)


In [8]:
# --- Tabla consolidada ---
df_resultados = pd.DataFrame(resultados)
tabla_mae = df_resultados.pivot(index=["origen", "regimen"], columns="modelo", values="mae").round(2)
tabla_mae = tabla_mae[["Persistencia", "XGBoost", "ARX+GARCH", "N-BEATSx", "N-HiTS"]]
for modelo in ["XGBoost", "ARX+GARCH", "N-BEATSx", "N-HiTS"]:
    tabla_mae[f"{modelo}_vs_persistencia"] = np.where(tabla_mae[modelo] < tabla_mae["Persistencia"], "gana", "pierde")
tabla_mae


,modelo,Persistencia,XGBoost,ARX+GARCH,N-BEATSx,N-HiTS,XGBoost_vs_persistencia,ARX+GARCH_vs_persistencia,N-BEATSx_vs_persistencia,N-HiTS_vs_persistencia
origen,regimen,,,,,,,,,
Origen 1,La Nina 2020-21,16.80,18.25,48.60,17.91,30.02,pierde,pierde,pierde,pierde
Origen 2,La Nina 2021-22,17.52,24.14,19.51,19.27,18.89,pierde,pierde,pierde,pierde
Origen 3,La Nina triple-dip -> neutral,39.75,59.60,47.59,38.59,38.90,pierde,pierde,gana,gana
Origen 4,El Nino 2023-24 (episodio completo),67.19,133.99,78.12,70.94,68.98,pierde,pierde,pierde,pierde
Origen 5,Neutral / La Nina debil 2024-25,101.25,135.47,102.59,104.66,102.42,pierde,pierde,pierde,pierde
Origen 6,El Nino 2026 (neutral->fuerte),56.40,61.39,55.83,49.19,46.80,pierde,gana,gana,gana


In [9]:
# --- Estabilidad de cada modelo a lo largo de los origenes/regimenes ---
for modelo in ["Persistencia", "XGBoost", "ARX+GARCH", "N-BEATSx", "N-HiTS"]:
    maes = df_resultados[df_resultados["modelo"] == modelo]["mae"]
    cv = maes.std() / maes.mean()
    print(f"{modelo:15s} MAE medio: {maes.mean():6.2f}  desv.std: {maes.std():6.2f}  CV: {cv:.2f}")

print()
ninos = df_resultados[df_resultados["regimen"].str.contains("Nino")]
ninas = df_resultados[df_resultados["regimen"].str.contains("Nina")]
for modelo in ["Persistencia", "XGBoost", "ARX+GARCH", "N-BEATSx", "N-HiTS"]:
    mae_nino = ninos[ninos["modelo"] == modelo]["mae"].mean()
    mae_nina = ninas[ninas["modelo"] == modelo]["mae"].mean()
    print(f"{modelo:15s} MAE promedio El Nino: {mae_nino:6.2f}   MAE promedio La Nina: {mae_nina:6.2f}   razon: {mae_nino/mae_nina:.2f}x")


Persistencia    MAE medio:  49.82  desv.std:  32.33  CV: 0.65
XGBoost         MAE medio:  72.14  desv.std:  51.61  CV: 0.72
ARX+GARCH       MAE medio:  58.71  desv.std:  28.54  CV: 0.49
N-BEATSx        MAE medio:  50.09  desv.std:  33.26  CV: 0.66
N-HiTS          MAE medio:  51.00  desv.std:  30.34  CV: 0.59

Persistencia    MAE promedio El Nino:  61.79   MAE promedio La Nina:  43.83   razon: 1.41x
XGBoost         MAE promedio El Nino:  97.69   MAE promedio La Nina:  59.36   razon: 1.65x
ARX+GARCH       MAE promedio El Nino:  66.98   MAE promedio La Nina:  54.57   razon: 1.23x
N-BEATSx        MAE promedio El Nino:  60.06   MAE promedio La Nina:  45.11   razon: 1.33x
N-HiTS          MAE promedio El Nino:  57.89   MAE promedio La Nina:  47.56   razon: 1.22x


In [10]:
# --- Guardar el resultado consolidado para el informe comparativo de OE2 ---
ruta_salida = RAIZ / "data" / "processed" / "resultados" / "walkforward_5origenes.csv"
df_resultados.to_csv(ruta_salida, index=False)
print("Guardado en:", ruta_salida)


Guardado en: C:\Users\mgdbj\xm-spot-price-predictor\data\processed\resultados\walkforward_5origenes.csv
